# scRareBench | High-Level scVI | Dataset 5 — NYGC / Seurat v4 PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

**High-level workflow**

`load_dataset(5) → MethodSpec(scVI) → benchmark_method() → scIB + rare-cell evaluation → multi-seed HTML/ZIP`

This notebook uses the **pinned scRareBench v0.10.6** source commit
`cfdb720f5488b9ceef9a225b35fc7558bcb0251c` and `scvi-tools==1.4.3`.

## Dataset 5 — verified package contract

| Field | Value |
|---|---|
| Registry key | `nygc_seurat_v4_pbmc` |
| Source | NYGC / Seurat v4 CITE-seq PBMC (official scverse example H5AD) |
| Source cells | 161,764 |
| **Post-QC benchmark cells** | **152,145** |
| Genes | 20,729 |
| Evaluation-only label | `celltype.l2` (30 categories) |
| Integration/evaluation batch | `orig.ident` (24 batches) |
| Canonical raw RNA counts | **`adata.X`** (`count_layer` is empty/None) |
| scIB reference HVGs | `evaluation_batch` |
| Rare-scenario metadata | 9 provisional six-state rows; **not** the paper-validated Dataset 0 annotations |

**Important Dataset 5 differences:**

- `load_dataset(5)` applies the **audited official scvi-tools PBMC QC** in memory, retaining 152,145 of 161,764 cells. Do **not** repeat QC in this notebook.
- The canonical RNA counts are already in `adata.X`. Unlike Dataset 3/4, **do not use `layers["counts"]` or `adata.raw.X`**.
- The 4,000 method-side HVGs are selected with Seurat-v3 `span=0.3` and `batch_key="orig.ident"` (the Dataset 5 audit passed this batch-aware selection).
- The scVI model receives only `orig.ident` as a batch covariate. Biological labels and registered rare-scenario metadata are **never used for training/HVG selection**.
- The registered Dataset 5 rare annotations are **provisional**, and not all six scenarios are represented. Interpretation must preserve this limitation.

The dataset preparation, rare-scenario annotation, reference scIB preprocessing, evaluation, and reports are owned by **scRareBench**. The notebook only implements the method-specific scVI adapter.

**Runtime:** select a GPU and preferably **High-RAM** in Google Colab. Dataset 5 is larger than Dataset 4; the 3-seed run can take substantial time and memory. For a smoke test, change `METHOD_SEEDS` to `[42]` first.

## 1. Install the pinned scRareBench and scVI runtime

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
REPO = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)

# Install the exact benchmark package first without replacing Colab's
# ABI-sensitive scientific stack; then let the package bootstrap dependencies.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", REPO,
])

METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)

from scrarebench.runtime import setup_runtime, print_install_report

install_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("scvi",),
    quiet=False,
)
print_install_report(install_report)


## 2. Load Dataset 5 through scRareBench (including package-owned QC)

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset, scenario_table_from_adata

adata = load_dataset(5)
info = dataset_info(adata)

# Fail-fast dataset contract checks. No manual QC or scenario attachment.
expected = {
    "dataset_key": "nygc_seurat_v4_pbmc",
    "label_key": "celltype.l2",
    "batch_key": "orig.ident",
    "scib_hvg_batch_mode": "evaluation_batch",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Dataset 5 contract mismatch for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

# The metadata serializes None as an empty string for count_layer.
if info.get("count_layer") not in (None, ""):
    raise RuntimeError(
        "Dataset 5 must use raw counts in adata.X, not an AnnData layer; "
        f"observed count_layer={info.get('count_layer')!r}"
    )

if adata.shape != (152_145, 20_729):
    raise RuntimeError(
        f"Dataset 5 post-QC shape changed: {adata.shape}; "
        "expected (152145, 20729)."
    )
if not adata.obs_names.is_unique or not adata.var_names.is_unique:
    raise RuntimeError("Dataset 5 cell/gene identifiers must be unique.")
if adata.obs["orig.ident"].astype(str).nunique() != 24:
    raise RuntimeError("Expected 24 Dataset 5 batches.")
if adata.obs["celltype.l2"].astype(str).nunique() != 30:
    raise RuntimeError("Expected 30 Dataset 5 biological label categories.")

prep = adata.uns.get("scrarebench_dataset_preparation", {})
if prep.get("count_source") != "adata.X":
    raise RuntimeError("Package-owned Dataset 5 canonical count source changed.")
if prep.get("source_cells") != 161_764 or prep.get("analysis_cells") != 152_145:
    raise RuntimeError("Package-owned Dataset 5 official QC counts changed.")
if prep.get("retained_cell_order_sha256") != (
    "830dd8a9ebb828d39dbd8cacd6a426edaf62385e9df4009424edc930589b3ec0"
):
    raise RuntimeError("Dataset 5 post-QC cell identity/order hash changed.")

scenario_table = scenario_table_from_adata(adata)
if scenario_table is None or len(scenario_table) != 9:
    raise RuntimeError(
        "Dataset 5 provisional registered six-state scenario table is missing "
        f"or changed (observed rows: {0 if scenario_table is None else len(scenario_table)}; expected 9)."
    )

print("scRareBench:", scrarebench.__version__)
print(adata)
print("\nDataset contract:")
for key in ("dataset_key", "label_key", "batch_key", "count_layer", "scib_hvg_batch_mode", "benchmark_ready"):
    print(f"  {key}: {info.get(key)!r}")
print("Official QC:", prep.get("qc", {}))
print("Registered provisional rare-scenario rows:", len(scenario_table))
print(scenario_table["scenario"].value_counts().sort_index())


## 3. scVI configuration

In [ ]:
# Method seeds vary; benchmark/evaluation randomness remains fixed.  [42, 123, 2026]
METHOD_SEEDS = [2026]  # First smoke run: [42]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    # Audited Dataset 5 method-side feature-selection policy.
    "n_hvg": 4000,
    "hvg_span": 0.3,
    "hvg_batch_key": "orig.ident",

    # scVI architecture and training (matched to Dataset 4 scVI notebook).
    "n_hidden": 128,
    "n_latent": 30,
    "n_layers": 2,
    "dropout_rate": 0.10,
    "dispersion": "gene-batch",
    "gene_likelihood": "nb",
    "max_epochs": 200,
    "batch_size": 256,
}

OUTPUT_DIR = "/content/scrarebench_results/dataset5_scvi"


## 4. User-owned scVI method adapter

This is the **only method-specific** section. It selects the audited batch-aware
HVGs from **`adata.X`**, creates a 4,000-gene scVI input, trains scVI using
`orig.ident`, and returns a 30-dimensional latent with explicit barcode
validation. No biological labels, rare-scenario annotations, or benchmark
metrics are passed to the model.

Because the original Dataset 5 count matrix is large, the runner subsets a
**copy of the HVG-only AnnData** for scVI; it does not normalize or overwrite
the canonical raw counts in `adata.X`.


In [ ]:
import gc

import numpy as np
import scanpy as sc
import scvi
import torch

from scrarebench import MethodOutput, dataset_info


def run_scvi(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info.get("count_layer") or None

    if method_info.get("dataset_key") != "nygc_seurat_v4_pbmc":
        raise RuntimeError("This scVI runner is restricted to Dataset 5.")
    if batch_key != "orig.ident":
        raise RuntimeError(
            f"Unexpected Dataset 5 batch key {batch_key!r}; expected 'orig.ident'."
        )
    if count_layer is not None:
        raise RuntimeError(
            "Dataset 5 canonical counts must be in adata.X; "
            f"unexpected count_layer={count_layer!r}."
        )
    if method_info.get("scib_hvg_batch_mode") != "evaluation_batch":
        raise RuntimeError("Dataset 5 requires evaluation-batch-aware reference HVGs.")
    if config["hvg_batch_key"] != batch_key:
        raise RuntimeError("Dataset 5 method HVGs must use orig.ident batches.")
    if not method_adata.obs_names.is_unique:
        raise RuntimeError("Dataset 5 cell barcodes must be unique.")
    if not torch.cuda.is_available():
        raise RuntimeError(
            "A CUDA GPU is required for this large scVI run. "
            "Select Runtime → Change runtime type → GPU in Colab."
        )

    original_barcodes = method_adata.obs_names.astype(str).copy()

    # Method-side HVG selection on *raw counts in X*; no biological label input.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=None,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        batch_key=batch_key,
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    if "highly_variable" not in method_adata.var:
        raise RuntimeError("Seurat-v3 HVG selection did not return a mask.")

    hvg_mask = (
        method_adata.var["highly_variable"]
        .fillna(False)
        .to_numpy(dtype=bool)
    )
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    selected_hvg = int(hvg_mask.sum())
    if selected_hvg != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} donor/batch-aware HVGs, "
            f"selected {selected_hvg}."
        )

    # Subset only method-owned scVI input. The original X remains raw counts.
    scvi_adata = method_adata[:, hvg_mask].copy()
    subset_barcodes = scvi_adata.obs_names.astype(str)
    if not subset_barcodes.equals(original_barcodes):
        raise RuntimeError("HVG subsetting changed Dataset 5 cell identity/order.")

    scvi.settings.seed = int(seed)
    scvi.model.SCVI.setup_anndata(
        scvi_adata,
        layer=None,  # Critical: Dataset 5 canonical raw counts are in X.
        batch_key=batch_key,
    )

    model = scvi.model.SCVI(
        scvi_adata,
        n_hidden=int(config["n_hidden"]),
        n_latent=int(config["n_latent"]),
        n_layers=int(config["n_layers"]),
        dropout_rate=float(config["dropout_rate"]),
        dispersion=str(config["dispersion"]),
        gene_likelihood=str(config["gene_likelihood"]),
    )

    model.train(
        max_epochs=int(config["max_epochs"]),
        train_size=0.90,
        validation_size=0.10,
        batch_size=int(config["batch_size"]),
        early_stopping=True,
        early_stopping_patience=20,
        accelerator="gpu",
        devices=1,
    )

    latent = np.asarray(model.get_latent_representation(), dtype=np.float32)
    expected_shape = (method_adata.n_obs, int(config["n_latent"]))
    if latent.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected scVI latent shape {latent.shape}; expected {expected_shape}."
        )
    if not np.isfinite(latent).all():
        raise RuntimeError("scVI latent contains NaN or infinite values.")
    output_barcodes = scvi_adata.obs_names.astype(str)
    if not output_barcodes.equals(original_barcodes):
        raise RuntimeError("scVI output barcode order differs from Dataset 5.")

    print(
        f"scVI seed={seed} | cells={scvi_adata.n_obs:,} | "
        f"HVGs={selected_hvg:,} | batches={scvi_adata.obs[batch_key].nunique()} | "
        f"latent={latent.shape}"
    )

    del model, scvi_adata
    gc.collect()
    torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=output_barcodes,
        representation_key="X_scvi",
    )


## 5. Run the High-Level multi-seed benchmark

The benchmark package handles all reference scIB preprocessing, clustering,
rare-cell evaluation, HTML reporting, and ZIP creation. For Dataset 5 the
reference uses raw counts from `X` and `evaluation_batch` HVGs.

**Memory policy:** `copy_input_per_seed=False` avoids duplicating the large
152,145 × 20,729 full AnnData for every method/evaluation pass. The scVI runner
creates its own **HVG-only** training copy and does not mutate the raw-count
matrix. `overwrite=True` allows scRareBench to refresh evaluation-owned latent,
neighbor and clustering outputs between seeds.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scVI",
    runner=run_scvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=False,
    overwrite=True,
)


## 6. Inspect and validate completed seed outputs

In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Method:", result.method)
print("Seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Final delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(
        f"seed={seed} | alignment={run.alignment} | "
        f"representation={run.representation_key} | "
        f"report={run.interactive_report_path}"
    )
    if run.representation_key != "X_scvi":
        raise RuntimeError(f"Unexpected representation for seed {seed}.")
    if run.alignment.get("exact_order_match") is not True:
        raise RuntimeError(f"Seed {seed} did not preserve exact barcode order.")


## 7. Download the finalized results directly to your computer

This is a **separate cell** so a browser download issue cannot be confused with
an incomplete benchmark. The final archive already exists once Step 5 succeeds.
You can also use the Colab **Files** sidebar to download the ZIP directly.


In [ ]:
from pathlib import Path
import shutil
from zipfile import ZipFile

archive = Path(result.archive_path)
if not archive.is_file():
    raise FileNotFoundError(f"Benchmark final archive was not created: {archive}")

# Short path avoids some Colab frontend filename issues.
download_copy = Path("/content/scVI_dataset5_results.zip")
if archive.resolve() != download_copy.resolve():
    shutil.copy2(archive, download_copy)

with ZipFile(download_copy, "r") as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity check failed: {bad_member}")

print("Final ZIP:", download_copy)
print("Size (MB):", round(download_copy.stat().st_size / 1024**2, 2))

try:
    from google.colab import files
    files.download(str(download_copy))
except ImportError:
    print("Not running in Colab; file remains at:", download_copy)


## Validation status

This notebook was checked statically for:

- notebook JSON/nbformat validity and Python AST syntax;
- the pinned Dataset 5 source/QC, label, batch and count-source contract;
- no training use of `celltype.l2` or rare-scenario labels;
- donor/batch-aware Seurat-v3 4,000-HVG selection on `X`;
- scVI `layer=None` and `batch_key="orig.ident"`;
- 30-dimensional finite latent and exact barcode alignment;
- high-level scRareBench evaluation, scIB, and multi-seed finalization.

**Static validation is not a successful Colab execution.** For a smoke test,
set `METHOD_SEEDS = [42]`. Restore `[42, 123, 2026]` for final benchmarking.


In [ ]:

from pathlib import Path
from zipfile import ZipFile
import shutil

from scrarebench.delivery import finalize_multiseed_delivery
from google.colab import files

ROOT = Path("/content")
SEEDS = [42, 123, 2026]
WORKDIR = ROOT / "scvi_dataset5_merge_inputs"
WORKDIR.mkdir(parents=True, exist_ok=True)

reports = []
bundles = {}

# 1. Validate and extract per-seed HTML reports
for seed in SEEDS:
    archive = ROOT / f"scVI_dataset5_results_{seed}.zip"

    if not archive.is_file():
        raise FileNotFoundError(f"Missing: {archive}")

    with ZipFile(archive, "r") as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(
                f"Corrupted ZIP for seed {seed}: {bad}"
            )

        member = "reports/interactive_report.html"
        if member not in zf.namelist():
            raise RuntimeError(
                f"Seed {seed}: missing {member}. "
                "Expected a scRareBench single-seed bundle."
            )

        seed_dir = WORKDIR / f"seed_{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)

        report_path = seed_dir / "interactive_report.html"

        with zf.open(member) as source:
            with report_path.open("wb") as dest:
                shutil.copyfileobj(source, dest)

        reports.append(report_path)
        bundles[seed] = archive

    print(f"Seed {seed}: ZIP and HTML ready")

# 2. Official scRareBench multi-seed finalization
delivery = finalize_multiseed_delivery(
    reports=reports,
    output_dir=ROOT / "scvi_dataset5_merged",
    method_name="scVI",
    dataset_key="nygc_seurat_v4_pbmc",
    expected_seeds=SEEDS,
    bundles_by_seed=bundles,
    include_latents=False,
    require_all_expected=True,
)

# 3. Prepare final ZIP for direct download
final_zip = ROOT / "scVI_dataset5_MERGED_3seeds.zip"
shutil.copy2(delivery.archive_path, final_zip)

print("\nMERGE SUCCESSFUL")
print("Completed seeds:", delivery.completed_seeds)
print("HTML report:", delivery.report_path)
print("Final ZIP:", final_zip)
print("Size (MB):", round(final_zip.stat().st_size / 1024**2, 2))

# 4. Download to your computer
files.download(str(final_zip))
